# GetGTExTables — build the merged GTEx tables in `code/analysis_files/`

Cleaned from `ds_v_dge_scratch.ipynb`, which produced these four tables through
ad-hoc cells that wrote into the notebook's own directory; the files were then
moved to `code/analysis_files/` by hand.

| Output | Consumed by |
|---|---|
| `GTEx.psi.tsv.gz` | **Fig. 2c heatmap** (`Figure2_heatmap_helpers.R`) |
| `GTEx.psi_pvals.tsv.gz` | supplementary / exploratory |
| `GTEx.exp_pvals.tsv.gz` | supplementary / exploratory |
| `GTEx.cluster_counts.tsv.gz` | supplementary / exploratory |

Each table merges 1,225 pairwise tissue comparisons on
`intron, cluster, itype, ctype, gene_name, gene_id`. Where a tissue appears in
more than one comparison, its values are collapsed with the **median**.

---

### The two source directories, and why they differ

`ds_v_dge_scratch.ipynb` did not read all four tables from the same place:

* `GTEx.psi.tsv.gz` was built from **`code/tmp/ds_v_dge/`** (2024-11-19)
* the other three from **`code/results/ds_v_dge_confounder/tables/`** (2025-03-09),
  the output of `rules/ds-dge.smk :: PrepareTablesForHeatmap_generalized`

They hold the same 1,225 comparisons with identical row and column counts, but
**different values** — median |Δ PSI| = 0.002, max 0.13, correlation 0.9997.

The reason: the PSI in these tables is **not raw PSI**. It is the per-group PSI
fitted by `leafcutter_ds`, pulled out of the comparison `.rds` by
`scripts/PrepareHeatmapTables_generalized.py`. The confounder run adds a
per-sample covariate — `log(unproductive junction reads / total junction reads)`,
appended to the group file by `scripts/prepare_groups_with_counfounder.py` — so
the fitted PSI shifts slightly. No samples are dropped; the model changes.

So `tmp/ds_v_dge/` is the pre-confounder run and
`results/ds_v_dge_confounder/tables/` is the confounder-corrected one.

**Both are built below**, to separate files, so Fig. 2c can be drawn either way
and compared:

| Output | Source | Used by |
|---|---|---|
| `GTEx.psi.tsv.gz` | `tmp/ds_v_dge/` (pre-confounder) | Fig. 2c as published |
| `GTEx.psi.confounder.tsv.gz` | rule output (confounder-corrected) | Fig. 2c_confounders |


In [ ]:
import os

import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = '/project/yangili1/cfbuenabadn/leafcutter2_paper'

# Pre-confounder run (2024-11-19) -- the source of the published GTEx.psi.tsv.gz
TABLES_TMP_DIR = f'{BASE}/code/tmp/ds_v_dge'
# Confounder-corrected run (2025-03-09), from
# rules/ds-dge.smk :: PrepareTablesForHeatmap_generalized
TABLES_RULE_DIR = f'{BASE}/code/results/ds_v_dge_confounder/tables'

OUT_DIR = f'{BASE}/code/analysis_files'

# Extension -> output filename, per source directory.
TABLES = {
    'psi': 'GTEx.psi.tsv.gz',
    'psi_p': 'GTEx.psi_pvals.tsv.gz',
    'exp_p': 'GTEx.exp_pvals.tsv.gz',
    'cluster_counts': 'GTEx.cluster_counts.tsv.gz',
}

KEYS = ['intron', 'cluster', 'itype', 'ctype', 'gene_name', 'gene_id']

os.makedirs(OUT_DIR, exist_ok=True)
print('pre-confounder tables :', TABLES_TMP_DIR)
print('confounder tables     :', TABLES_RULE_DIR)
print('writing to            :', OUT_DIR)

In [ ]:
def list_comparisons(source_dir):
    """The pairwise tissue comparisons available, from the .psi.* filenames."""
    files_list = os.listdir(source_dir)
    return sorted({x.split('.psi.')[0] for x in files_list if '.psi.' in x})


def merge_dataframes(df1, df2):
    """Outer-merge two comparison tables on the intron keys.

    A tissue can appear in several comparisons. When that happens the merge
    produces `<tissue>_x` and `<tissue>_y`, which are collapsed to their median
    and the suffixed columns dropped.
    """
    df = pd.merge(df1, df2, left_on=KEYS, right_on=KEYS, how='outer')

    if any(x.endswith('_x') for x in df.columns):
        duplicate_tissues = [x.split('_x')[0] for x in df.columns if x.endswith('_x')]
        for tissue in duplicate_tissues:
            df[tissue] = list(np.array(df[[f'{tissue}_x', f'{tissue}_y']].median(axis=1)))

    return df[[x for x in df.columns if not (x.endswith('_x') or x.endswith('_y'))]]


def merge_all_dataframes(ext, comparisons, source_dir):
    """Merge every comparison's `.{ext}.tsv.gz` into one wide table."""
    df = pd.DataFrame(columns=KEYS)
    for comparison in tqdm(comparisons, position=0, leave=True, desc=ext):
        df_ = pd.read_csv(f'{source_dir}/{comparison}.{ext}.tsv.gz', sep='\t')
        df = merge_dataframes(df, df_)
    return df


def build_tables(source_dir, tables, suffix=''):
    """Build every table in `tables` from `source_dir`, writing into OUT_DIR.

    `suffix` is inserted before '.tsv.gz' so the confounder versions land beside
    the originals instead of overwriting them.
    """
    comparisons = list_comparisons(source_dir)
    print(f'{len(comparisons)} comparisons in {source_dir}')
    for ext, out_name in tables.items():
        df = merge_all_dataframes(ext, comparisons, source_dir)
        if suffix:
            out_name = out_name.replace('.tsv.gz', f'{suffix}.tsv.gz')
        out_path = f'{OUT_DIR}/{out_name}'
        df.to_csv(out_path, sep='\t', header=True, index=False)
        print(f'  {out_name:<34} {df.shape[0]:>8} rows x {df.shape[1]:>3} cols')
        del df

## Build the tables

Each table is a 1,225-way merge, so both cells are slow and memory-hungry.

In [ ]:
# Pre-confounder tables -- reproduces the published GTEx.*.tsv.gz
build_tables(TABLES_TMP_DIR, TABLES)

In [ ]:
# Confounder-corrected PSI only. Written as GTEx.psi.confounder.tsv.gz, which
# Figure2_heatmap.ipynb reads for the fig2c_confounders panel.
build_tables(TABLES_RULE_DIR, {'psi': 'GTEx.psi.tsv.gz'}, suffix='.confounder')